# Experiment 7: Constituency and Dependency Parsing

- **Constituency:** nested phrases (NP, VP, PP) from a Context-Free Grammar (CFG).
- **Ambiguity:** "I saw the man with the telescope" has 2 trees (who has the telescope?).
- **PCFG:** rules have probabilities; Viterbi picks the most likely tree.
- **Dependency:** each word points to its head word with a relation (nsubj, dobj, ...).

In [ ]:
# !pip install nltk spacy
# !python -m spacy download en_core_web_sm

In [ ]:
import nltk, spacy
from spacy import displacy
from nltk import CFG, PCFG, ChartParser, ViterbiParser

nlp = spacy.load('en_core_web_sm')

## A small grammar

In [ ]:
grammar = CFG.fromstring("""
    S   -> NP VP
    NP  -> Det N | Det Adj N | NP PP | Pron
    VP  -> V NP | VP PP
    PP  -> P NP
    Det -> 'the' | 'a'
    Adj -> 'detailed'
    N   -> 'student' | 'report' | 'man' | 'telescope' | 'library'
    Pron -> 'I'
    V   -> 'wrote' | 'saw'
    P   -> 'with' | 'in'
""")
parser = ChartParser(grammar)

tree = next(parser.parse('the student wrote a detailed report'.split()))
tree.pretty_print()

## Ambiguity: two valid trees

In [ ]:
sentence = 'I saw the man with the telescope'.split()
trees = list(parser.parse(sentence))
print('Number of trees:', len(trees))
for t in trees:
    t.pretty_print()

In [ ]:
# A wrong word order gets 0 trees
print(len(list(parser.parse('saw I man the'.split()))), 'trees')

## PCFG picks the most likely tree

In [ ]:
pcfg = PCFG.fromstring("""
    S   -> NP VP          [1.0]
    NP  -> Det N [0.5] | NP PP [0.2] | Pron [0.3]
    VP  -> V NP [0.7] | VP PP [0.3]
    PP  -> P NP           [1.0]
    Det -> 'the'          [1.0]
    N   -> 'man' [0.5] | 'telescope' [0.5]
    Pron -> 'I'           [1.0]
    V   -> 'saw'          [1.0]
    P   -> 'with'         [1.0]
""")
best = next(ViterbiParser(pcfg).parse(sentence))
print('Probability:', best.prob())
best.pretty_print()

## Dependency parsing with SpaCy

In [ ]:
doc = nlp('The researchers who developed the new model presented their results at the conference.')
for t in doc:
    print(f'{t.text:12} --{t.dep_:>6}--> {t.head.text}')

In [ ]:
displacy.render(doc, style='dep', jupyter=True, options={'compact': True})

## Subject-Verb-Object extraction

In [ ]:
for t in doc:
    if t.dep_ == 'nsubj':                       # found a subject
        verb = t.head
        for obj in verb.children:
            if obj.dep_ == 'dobj':              # its direct object
                print('Subject:', t.text, '| Verb:', verb.lemma_, '| Object:', obj.text)